# Build custom atac reference binning

The genome binning strategy used to build the ATAC reference MUST be the same used for the following preprocessing of the ATAC samples.

## Setting up reference from custom genome

The workflow may start from one of two staring point:
- a FASTA file index, usually ending with `.fai` suffix.
- a chromosome size `pandas.DataFrame`



Here is the formatting of the expected `.fai` file. We only require the first and the second columns of the `.fai` file, respectively the chromosome ID and the total length of the chromosome (in base pairs).

In [1]:
import pandas as pd

In [2]:
chrom_size_real_path = "/beegfs/scratch/ric.cosr/carlino.calogero/genomes/refdata-gex-GRCh38-2024-A/fasta/genome.fa.fai"


In [3]:
pd.read_table(chrom_size_real_path, header=None).head()

,0,1,2,3,4
0,chr1,248956422,8,60,61
1,chr10,133797422,253105714,60,61
2,chr11,135086622,389133104,60,61
3,chr12,133275309,526471180,60,61
4,chr13,114364328,661967755,60,61


Given the path to the `.fai` file, preprocessing can be performed as follow:

In [4]:
import spacenumbat
from spacenumbat.preprocessing import multiome_unpaired

In [5]:
minimal_chrom_size_df = multiome_unpaired.get_minimal_chrom_size_from_fasta_index(chrom_size_real_path)

In [6]:
minimal_chrom_size_df.head()

,CHROM,length
0,chr1,248956422
1,chr2,242193529
2,chr3,198295559
3,chr4,190214555
4,chr5,181538259


Once the `pd.DataFrame` with the size of each desired chromosome is built or imported, we can proceed with the binning.

You can use a custom binning of your choice and import it. If you want a custom binning with fixed bin size this can be imported or constructed with the provided function.

To build a fixed size binning of your choice, the following function can be used: 

In [7]:
bin_size = 220_000
current_binning = multiome_unpaired.get_custom_gtf_binning(minimal_chrom_size_df, 
                                                           bin_size=bin_size)


In [8]:
current_binning.head()

,bin_id,CHROM,start,end,width
chr1:0-220000,chr1:0-220000,1,0,220000,220000
chr1:220000-440000,chr1:220000-440000,1,220000,440000,220000
chr1:440000-660000,chr1:440000-660000,1,440000,660000,220000
chr1:660000-880000,chr1:660000-880000,1,660000,880000,220000
chr1:880000-1100000,chr1:880000-1100000,1,880000,1100000,220000


To import a custom binning just import it as a `pandas.DataFrame`. Here we import the genome binning used in the R implementation of `Numbat`.

In [9]:
numbat_bins_path = "/beegfs/scratch/ric.cosr/ric.cosr/InnovationLab/spatial_dataset/spatial_multiomics/numbat_bins.tsv"


In [10]:
current_binning = pd.read_table(numbat_bins_path,index_col=0)


In [11]:
current_binning.head()

,bin_id,CHROM,start,end,width
chr1:0-1042457,chr1:0-1042457,1,0,1042457,1042457
chr1:1042457-1265484,chr1:1042457-1265484,1,1042457,1265484,223027
chr1:1265484-1519859,chr1:1265484-1519859,1,1265484,1519859,254375
chr1:1519859-1826619,chr1:1519859-1826619,1,1519859,1826619,306760
chr1:1826619-2058465,chr1:1826619-2058465,1,1826619,2058465,231846


In [24]:
genomic_regions = current_binning.bin_id.unique().tolist()

In [25]:
genomic_regions

['chr1:0-1042457',
 'chr1:1042457-1265484',
 'chr1:1265484-1519859',
 'chr1:1519859-1826619',
 'chr1:1826619-2058465',
 'chr1:2058465-2280372',
 'chr1:2280372-2491263',
 'chr1:2491263-2817807',
 'chr1:2817807-3025418',
 'chr1:3025418-3230151',
 'chr1:3230151-3437000',
 'chr1:3437000-3645603',
 'chr1:3645603-4091348',
 'chr1:4091348-4312450',
 'chr1:4312450-4522391',
 'chr1:4522391-4733807',
 'chr1:4733807-4949751',
 'chr1:4949751-5167781',
 'chr1:5167781-5378577',
 'chr1:5378577-5586352',
 'chr1:5586352-5802602',
 'chr1:5802602-6013527',
 'chr1:6013527-6232553',
 'chr1:6232553-6468662',
 'chr1:6468662-6699855',
 'chr1:6699855-6922979',
 'chr1:6922979-7134271',
 'chr1:7134271-7347565',
 'chr1:7347565-7562625',
 'chr1:7562625-7781202',
 'chr1:7781202-8012370',
 'chr1:8012370-8246678',
 'chr1:8246678-8471404',
 'chr1:8471404-8698958',
 'chr1:8698958-8937804',
 'chr1:8937804-9174404',
 'chr1:9174404-9392992',
 'chr1:9392992-9634910',
 'chr1:9634910-9874006',
 'chr1:9874006-10127844',
 'chr

The above `genomic_regions` is a `List` that define the binning interval for the reference. The same binning should be passed to the `SpaceNumbat` pipeline. 

## Load reference dataset

We use the dataset [GSE184462](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE184462), that contains various samples collected from different tissues of multiple donors. We specifically use the supplementary file used in the linked GEO records.

First, we load and process the metadata:

In [12]:
atlas_samples_dir = "/beegfs/scratch/ric.cosr/ric.cosr/InnovationLab/spatial_dataset/spatial_multiomics/atac_reference_building/GSE184462_RAW"
atlas_metadata_path = "/beegfs/scratch/ric.cosr/ric.cosr/InnovationLab/spatial_dataset/spatial_multiomics/atac_reference_building/GSE184462_metadata.tsv"


In [13]:
metadata_atlas = pd.read_table(atlas_metadata_path)
metadata_atlas.columns = [str.lower("_".join(i.split(" "))) for i in metadata_atlas.columns]
metadata_atlas["barcode"] = [i.split("+")[1] for i in metadata_atlas.cellid]

We then extract the list of the genomic bin we calculated before and load the data for which metadata are available.

In [14]:
import os
import tqdm
from spacenumbat.preprocessing.multiome_unpaired import get_atac_binning
from spacenumbat.preprocessing.build_atac_ref import build_atac_reference
import anndata as ad

In [ ]:
atlas_dict = {}

for frag_file in tqdm.tqdm(os.listdir(atlas_samples_dir)):
    full_frag_path = os.path.join(atlas_samples_dir, frag_file)
    frag_file_split = frag_file.split("_")
    gsm_name = frag_file_split[0]
    sample_name = "_".join(frag_file_split[1:-2])
    current_sample_df = metadata_atlas[metadata_atlas["tissue"] == sample_name].loc[:,['sample', 
                                                                                       'tissue',
                                                                                       'cell_type', 
                                                                                       'life_stage',
                                                                                       'barcode']]
    if current_sample_df.shape[0] > 0:
    
        current_barcodes = current_sample_df["barcode"].tolist()
    
        atlas_dict[sample_name] = get_atac_binning(fragments_path=full_frag_path, 
                                                   genomic_regions=genomic_regions,
                                                   barcodes=current_barcodes,
                                                   counting_strategy="fragment",
                                                   min_num_fragments=0)

In [ ]:
for current_key, current_adata in atlas_dict.items():

    current_adata.obs = pd.merge(atlas_dict[current_key].obs, 
                                 metadata_atlas[metadata_atlas["tissue"] == current_key].loc[:,['sample', 
                                                                                                'tissue',
                                                                                                'cell_type', 
                                                                                                'life_stage',
                                                                                                'barcode']],
                                 left_index=True,
                                 right_on="barcode").set_index("barcode", drop=False)

for current_key, current_adata in atlas_dict.items():
    current_adata.obs["cellid"] = current_adata.obs["tissue"] + "_" + current_adata.obs["barcode"]
    current_adata.obs = current_adata.obs.set_index("cellid", drop=True)

In [ ]:
full_ad = ad.concat(atlas_dict, join="outer")

In [ ]:
full_ad.obs[["tissue_ref", "donor"]] = full_ad.obs["tissue"].str.rsplit("_", n=1, expand=True)


In [ ]:
atlas_adata_path = "/beegfs/scratch/ric.cosr/ric.cosr/InnovationLab/spatial_dataset/spatial_multiomics/atac_reference_building/atlas_adata.h5ad"
full_ad.write_h5ad(atlas_adata_path)

Our anndata at this point is as follow:

In [16]:
full_ad = ad.read_h5ad(atlas_adata_path)



In [17]:
full_ad

AnnData object with n_obs × n_vars = 484608 × 12145
    obs: 'n_fragment', 'frac_dup', 'frac_mito', 'sample', 'tissue', 'cell_type', 'life_stage', 'barcode', 'tissue_ref', 'donor'

In [18]:
full_ad.obs.head()

,n_fragment,frac_dup,frac_mito,sample,tissue,cell_type,life_stage,barcode,tissue_ref,donor
cellid,,,,,,,,,,
artery_aorta_SM-JF1NU_AAACGCAAGCAAAGCCCACGAC,5074,0.389020,0.021597,artery_aorta_SM-JF1NU_1,artery_aorta_SM-JF1NU,Vascular Smooth Muscle 1,Adult,AAACGCAAGCAAAGCCCACGAC,artery_aorta,SM-JF1NU
artery_aorta_SM-JF1NU_AAACGCAAGCAAAGCGGGAGCT,6583,0.384553,0.033759,artery_aorta_SM-JF1NU_1,artery_aorta_SM-JF1NU,Vascular Smooth Muscle 1,Adult,AAACGCAAGCAAAGCGGGAGCT,artery_aorta,SM-JF1NU
artery_aorta_SM-JF1NU_AAACGCAAGCAAAGGAACAGAC,4502,0.388786,0.094347,artery_aorta_SM-JF1NU_1,artery_aorta_SM-JF1NU,Fibroblast (General),Adult,AAACGCAAGCAAAGGAACAGAC,artery_aorta,SM-JF1NU
artery_aorta_SM-JF1NU_AAACGCAAGCAAAGGGATGCCA,6101,0.400706,0.001146,artery_aorta_SM-JF1NU_1,artery_aorta_SM-JF1NU,"Macrophage (General,Alveolar)",Adult,AAACGCAAGCAAAGGGATGCCA,artery_aorta,SM-JF1NU
artery_aorta_SM-JF1NU_AAACGCAAGCAACCATGCATGA,2382,0.386880,0.099433,artery_aorta_SM-JF1NU_1,artery_aorta_SM-JF1NU,T Lymphocyte 1 (CD8+),Adult,AAACGCAAGCAACCATGCATGA,artery_aorta,SM-JF1NU


In [19]:
full_ad.var.head()

""
chr1:0-1042457
chr1:1042457-1265484
chr1:1265484-1519859
chr1:1519859-1826619
chr1:1826619-2058465


In [20]:
(atac_reference,
 atac_manifest,
 atac_bin_gtf,
 atac_diag) = build_atac_reference(full_ad,
                                   donor_col='donor',
                                   tissue_col='tissue_ref',
                                   cell_type_col='cell_type',
                                   life_stage_col='life_stage',
                                   life_stage='Adult',
                                   min_cells=50,
                                   min_fragments=50_000,
                                   min_donors=1,
                                   min_tissue_donors=2,
                                   subtype_ratio_threshold=1.5,
                                   tissue_ratio_threshold=2.0,
                                   return_diagnostics=True)

In [22]:
atac_reference.columns = [str.lower("_".join(i.split(" "))) for i in atac_reference.columns]

In [23]:
atac_reference.head()

,adipocyte,airway_goblet_cell,alveolar_capillary_endothelial_cell,alveolar_type_1_(at1)_cell,alveolar_type_2_(at2)_cell,atrial_cardiomyocyte,basal_epidermal_(skin),basal_epithelial_(mammary),cd4_t,cd8_t,...,thyroid_follicular_cell,transitional_zone_cortical_cell,tuft_cell,type_i_skeletal_myocyte,type_ii_skeletal_myocyte,vascular_smooth_muscle_1,vascular_smooth_muscle_2,ventricular_cardiomyocyte,zona_fasciculata_cortical_cell,zona_glomerulosa_cortical_cell
chr1:0-1042457,0.000252,0.000303,0.000260,0.000239,0.000287,0.000303,0.000412,0.000543,0.000604,0.000342,...,0.000262,0.000655,0.000747,0.000377,0.000387,0.000396,0.000520,0.000335,0.000543,0.000573
chr1:1042457-1265484,0.000163,0.000513,0.000151,0.000161,0.000166,0.000109,0.000321,0.000431,0.000348,0.000250,...,0.000149,0.000172,0.000483,0.000121,0.000117,0.000147,0.000120,0.000111,0.000165,0.000167
chr1:1265484-1519859,0.000371,0.000483,0.000186,0.000248,0.000243,0.000168,0.000477,0.000584,0.000351,0.000281,...,0.000231,0.000528,0.000419,0.000412,0.000393,0.000326,0.000304,0.000241,0.000513,0.000500
chr1:1519859-1826619,0.000365,0.000281,0.000180,0.000141,0.000159,0.000143,0.000380,0.000349,0.000285,0.000270,...,0.000241,0.000500,0.000229,0.000286,0.000283,0.000249,0.000172,0.000191,0.000586,0.000450
chr1:1826619-2058465,0.000196,0.000201,0.000136,0.000156,0.000139,0.000145,0.000254,0.000244,0.000259,0.000182,...,0.000157,0.000369,0.000196,0.000174,0.000183,0.000144,0.000097,0.000184,0.000418,0.000337
